> 在线讲义：[第 9 章 · 仿真、数据与部署闭环](https://wikieden.github.io/robot-architecture/lessons/09-sim-data-deploy.html) · 本书《机器人软硬件架构》

# 第 9 章 · 仿真、数据与部署闭环

配套讲义：仓库里 `robot-system-architecture/notes/lessons/09-sim-data-deploy.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（成功率是动画里的玩具模型；系统辨识的一维模型和数据量的条数是编的）

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第 3 步 / 动画 | 窄训练 vs 宽训练 | `success(mu, seen, lo, hi)` | μ = 0.5：窄 97%、宽 85%；μ = 0.9：窄 0%、宽 85% |
| 公式表 域随机化目标 | $\mathbb{E}_{\xi}[J(\pi;\xi)]$ | `expected_return` | 0.2–1.4 里随机 2000 个：窄 24%、宽 83% |
| 公式表 系统辨识 | $\arg\min_\xi \sum\lVert s_{t+1} - f(s_t, a_t;\xi)\rVert^2$ | `sysid_damping()` | 真值 0.8，估出 0.812 |
| 公式表 数据量 | 条数 × 秒 / 3600 | `dataset_hours(200, 45)` | 2.5 小时 |

成功率函数照动画里的玩具模型；系统辨识的一维模型和数据量估算的条数是编的。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `success`**：动画的玩具成功率：离最近一个训练摩擦越远越差；训练范围越宽，峰值越低（0.97 − 0.12 × 宽度 / 1.2）。

In [ ]:
def success(mu: float, seen: np.ndarray, lo: float, hi: float) -> float:
    """动画的玩具成功率：离最近一个训练摩擦越远越差；训练范围越宽，峰值越低（0.97 − 0.12 × 宽度 / 1.2）。"""
    width = hi - lo
    peak = 0.97 - 0.12 * width / 1.2
    sig = 0.12
    best = float(np.max(np.exp(-((mu - seen) ** 2) / (2 * sig**2))))
    cover = min(1.0, len(seen) / 12)
    return peak * cover * best

**函数 `expected_return`**：域随机化目标里的期望：把策略放到一批测试参数里，求平均成功率。

In [ ]:
def expected_return(seen: np.ndarray, lo: float, hi: float, test: np.ndarray) -> float:
    """域随机化目标里的期望：把策略放到一批测试参数里，求平均成功率。"""
    return float(np.mean([success(m, seen, lo, hi) for m in test]))

**函数 `sysid_damping`**：一维系统辨识：s_{t+1} = s_t + dt (a_t − ξ s_t) + 噪声，用最小二乘估出 ξ。

In [ ]:
def sysid_damping(xi_true: float = 0.8, n: int = 500, dt: float = 0.01, seed: int = 0) -> float:
    """一维系统辨识：s_{t+1} = s_t + dt (a_t − ξ s_t) + 噪声，用最小二乘估出 ξ。"""
    rng = np.random.default_rng(seed)
    s = np.zeros(n + 1)
    a = rng.uniform(-1, 1, n)
    for t in range(n):
        s[t + 1] = s[t] + dt * (a[t] - xi_true * s[t]) + rng.normal(0, 1e-4)
    # 把 (s_{t+1} − s_t)/dt − a_t = −ξ s_t 写成 y = X ξ
    y = (s[1:] - s[:-1]) / dt - a
    X = -s[:-1]
    return float(X @ y / (X @ X))

**函数 `dataset_hours`**：数据量估算：小时数 = episode 数 × 每条秒数 / 3600。

In [ ]:
def dataset_hours(episodes: int, seconds_each: float) -> float:
    """数据量估算：小时数 = episode 数 × 每条秒数 / 3600。"""
    return episodes * seconds_each / 3600

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第三步 / 动画：窄训练（只在 μ = 0.5）vs 宽训练（0.2–1.4 均匀取 13 个）")
narrow = np.full(13, 0.5)
wide = np.linspace(0.2, 1.4, 13)
for mu in (0.5, 0.9):
    print(f"  真机 μ = {mu}：窄 {success(mu, narrow, 0.5, 0.5):.0%}，宽 {success(mu, wide, 0.2, 1.4):.0%}")

In [ ]:
print("\n公式表：域随机化目标 E_ξ[J(π; ξ)]——在 0.2–1.4 里随机抽 2000 个真机摩擦求平均")
test = np.random.default_rng(0).uniform(0.2, 1.4, 2000)
print(f"  窄训练平均 {expected_return(narrow, 0.5, 0.5, test):.0%}，宽训练平均 {expected_return(wide, 0.2, 1.4, test):.0%}")

In [ ]:
print("\n公式表：系统辨识（编的一维模型，真实阻尼 ξ = 0.8）")
print(f"  最小二乘估出 ξ̂ = {sysid_damping():.3f}")

In [ ]:
print("\n公式表：数据量估算（编的：200 条 × 每条 45 秒）")
print(f"  {dataset_hours(200, 45):.2f} 小时")

## 3. 代码里才看得到的

- 这里的成功率是动画用的**玩具公式**，不是真机测出来的；它只用来演示"窄而高 vs 宽而平"的形状。
- 系统辨识把模型改写成 $y = X\xi$ 的形式后，最小二乘一行就解出来：`X @ y / (X @ X)`。估计值 0.812 和真值差一点，是因为加了噪声。

## 4. 自己改一改

- 把宽训练的范围改成 0.3–0.9，看真机 μ = 0.9 时成功率掉多少。
- 把 `sysid_damping` 的噪声从 1e-4 加到 1e-3，看估计值偏多少。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 仿真练、遥操作教、日志复盘、OTA 升级，转起来就是数据飞轮。
> 仿真不准就让它"随机地不准"；数据不记相机在哪、什么时候，就等于白采。